In [ ]:
# needed libraries

import os

import json

import requests

import pandas as pd

from datetime import datetime, timedelta

# libraries related to airflow

from airflow import DAG

from airflow.models import Variable

from airflow.utils.dates import days_ago

from airflow.operators.python import PythonOperator

# scikit learn libraries

from sklearn.model_selection import cross_val_score

from sklearn.linear_model import LinearRegression

from sklearn.tree import DecisionTreeRegressor

from sklearn.ensemble import RandomForestRegressor

from joblib import dump


# configurations

API_KEY = Variable.get("openweathermap_api_key")

CITIES = Variable.get("cities", deserialize_json=True)

# mount points by docker compose (make it faster to use in code)

RAW = "/app/raw_files"

CLEAN = "/app/clean_data"


# task 1 about retrieve data from open weather map

def get_data_from_api():

    """fetch current weather for cities, save as json file"""

    results = []

    for city in CITIES:

        # the openweathermap request url

        url = (

            f"https://api.openweathermap.org/data/2.5/weather"

            f"?q={city}&appid={API_KEY}"

        )

        response = requests.get(url)

        if response.status_code == 200:

            results.append(response.json())

        else:

            print(f"Error for {city}: {response.status_code}")

    # get a pseocifc date pattern

    now = datetime.now().strftime("%Y-%m-%d %H:%M")

    # build full file path, join folder raw with json file

    filepath = os.path.join(RAW, f"{now}.json")

    # save city responses into a json file

    with open(filepath, 'w') as f:

        json.dump(results, f)

    print(f"Saved {len(results)} records to {filepath}")

# transform data into csv

def transform_data_into_csv(n_files=None, filename='data.csv'):

    parent_folder = RAW

    files = sorted(os.listdir(parent_folder), reverse=True)

    if n_files:

        files = files[:n_files]

    dfs = []

    for f in files:

        # add this to skip non json files

        if not f.endswith(".json"):

            continue

        with open(os.path.join(parent_folder, f), 'r') as file:

            try:

                data_temp = json.load(file)

                # skip bad files

            except json.JSONDecodeError:

                continue

        for data_city in data_temp:

            dfs.append(

                {

                    'temperature': data_city['main']['temp'],

                    'city': data_city['name'],

                    'pression': data_city['main']['pressure'],

                    'date': f.split('.')[0],

                }

            )

    if not dfs:

        print('no data available yet')

        return

    df = pd.DataFrame(dfs)

    print("\n", df.head(10))

    df.to_csv(os.path.join(CLEAN,filename), index=False)

# concatenate 20 most recent json files into data.csv

def transform_last_20():

    transform_data_into_csv(n_files=20, filename='data.csv')

# conatenate all JSON files onto fulldata.csv

def transform_all():

    transform_data_into_csv(n_files=None, filename='fulldata.csv')

# model scoring

def compute_model_score(model,x, y):

    cross_validation = cross_val_score(

        model, x, y, cv=3, scoring='neg_mean_squared_error'

    )

    model_score = cross_validation.mean()

    return model_score

# train and save the model functions

def train_and_save_model(model, x, y, path_to_model=f"{CLEAN}/best_model.pickle"):

    model.fit(x,y)

    print(str(model), 'saved at ', path_to_model)

    dump(model, path_to_model)

# prepare data for training

def prepare_data(path_to_data=f"{CLEAN}/fulldata.csv"):

    df = pd.read_csv(path_to_data)

    df = df.sort_values(['city', 'date'], ascending=True)

    dfs = []

    for c in df['city'].unique():

        df_temp = df[df['city'] == c]

        df_temp['target'] = df_temp['temperature'].shift(1)

        # creat 9 features

        for i in range(1,10):

            df_temp.loc[:, 'temp_m-{}'.format(i)

                        ] = df_temp['temperature'].shift(-i)

        df_temp = df_temp.dropna()

        dfs.append(df_temp)

        # concatinate datsets

    df_final = pd.concat(dfs, axis=0, ignore_index=False)

    # rmeove date column as it cant be used as numeric for trinaing

    df_final = df_final.drop(['date'], axis=1)

    df_final = pd.get_dummies(df_final)

    features = df_final.drop(['target'], axis=1)

    target = df_final['target']

    return features, target


# train linear regersison, ** context collects all keyword arguments into dicitonary cntext

def train_linear_regression(**context):

    x, y = prepare_data()

    # use compute_model_score to run cross validaiton

    score = compute_model_score(LinearRegression(), x, y)

    context['ti'].xcom_push(key='score_lr', value=score)

    print(f'Linear regression score: {score}')

# train decision tree

def train_decision_tree(**context):

    x, y = prepare_data()

    score = compute_model_score(DecisionTreeRegressor(), x, y)

    context['ti'].xcom_push(key='score_dt', value=score)

    print(f'decision tree score: {score}')

# the random forest regressor model

def train_random_forest(**context):

    x, y = prepare_data()

    score = compute_model_score(RandomForestRegressor(), x, y)

    context['ti'].xcom_push(key='score_rf', value=score)

    print(f'forest score: {score}')


# select and save best model

# select xcom scores, retrina modle on all data

def select_best_model(**context):

    ti = context['ti']

    scores = {

        'lr':ti.xcom_pull(key='score_lr', task_ids='train_linear_regression'),

        'dt':ti.xcom_pull(key='score_dt', task_ids='train_decision_tree'),

        'rf':ti.xcom_pull(key='score_rf', task_ids='train_random_forest'),

    }

    best = max(scores, key=scores.get)

    print(f"Scores: {scores} --> best: {best}")


    x, y = prepare_data()

    models = {

        'lr': LinearRegression(),

        'dt': DecisionTreeRegressor(),

        'rf': RandomForestRegressor(),

    }

    train_and_save_model(models[best], x, y)

# dag definisiton

with DAG(

    dag_id='weather_dag',

    description='fetch weather data, transform, train and select best model',

    tags=['exam', 'datascientist', 'weather'],

    # to run every minute

    schedule_interval = "* * * * *",

    start_date=days_ago(0, minute=1),

    catchup=False,

    default_args={

        'owner':'airflow',

        'retries': 1,

        'retry_delay': timedelta(seconds=30),

    },

) as dag:

    # the task instiances

    # retrieve task

    get_data = PythonOperator(

        task_id = 'get_data_from_api',

        python_callable = get_data_from_api,

    )

    # transform last 20 files

    transform_last_20_task = PythonOperator(

        task_id='transform_last_20',

        python_callable=transform_last_20,

    )

    # transform all files

    transform_all_task = PythonOperator(

        task_id='transform_all',

        python_callable=transform_all,

    )

    # train 3 models

    train_lr = PythonOperator(

        task_id='train_linear_regression',

        python_callable=train_linear_regression,

    )

    train_dt = PythonOperator(

        task_id='train_decision_tree',

        python_callable=train_decision_tree,

    )

    train_rf = PythonOperator(

        task_id='train_random_forest',

        python_callable=train_random_forest,

    )


    # slecte and save the best one

    select_best = PythonOperator(

        task_id='select_best_model',

        python_callable=select_best_model,

    )


    # the dependencies and architecure

    get_data >> [transform_last_20_task, transform_all_task]

    transform_all_task >> [train_lr, train_dt, train_rf]

    [train_lr, train_dt, train_rf] >> select_best